---
Importing Libraries

In [ ]:
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures, OrdinalEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.experimental import enable_hist_gradient_boosting
from sklearn.calibration import CalibratedClassifierCV

from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from scipy.stats import loguniform

from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    confusion_matrix, classification_report)

----------------------------------------------------
Loading features and target variable into dataframes

In [ ]:
X = pd.read_csv("values.csv")
y = pd.read_csv("labels.csv")

In [ ]:
X

In [ ]:
y

In [ ]:
X.columns

In [ ]:
y.columns

---------------------------
Checking for duplicate rows

In [ ]:
X.duplicated(subset=["patient_id"], keep='first').sum()

---
Creating a dataframe merging features and target variable 

In [ ]:
df = X.merge(y, on="patient_id", how="inner").drop_duplicates(subset=["patient_id"]).reset_index(drop=True)
df
target_col = "heart_disease_present"
id_col = "patient_id"

---
features and target variable are synced on "patient_id", therefore removing "patient_id" column from X and y

In [ ]:
X.drop(columns = ["patient_id"], inplace = True)
X

In [ ]:
y.drop(columns = ["patient_id"], inplace = True)
y

---
Checking for missing values

In [ ]:
X.isna().sum()

In [ ]:
y.isna().sum()

---
Converting nominal categorical columns to categorical data type

In [ ]:
X[['thal', 'chest_pain_type', 'resting_ekg_results']] = X[['thal', 'chest_pain_type', 'resting_ekg_results']].astype('category')

In [ ]:
X.dtypes

Verifying Class balance

In [ ]:
y.value_counts()

- not severely skewed
- use stratified splits
- no oversampling necessary
- optimize for AUPRC and sensitivity for clinical priority

---
Check ranges of categorical variables

In [ ]:
val_counts = []
for col in ['slope_of_peak_exercise_st_segment', 'thal', 'chest_pain_type', 'num_major_vessels', 'fasting_blood_sugar_gt_120_mg_per_dl', 'resting_ekg_results', 'sex', 'exercise_induced_angina']:
    val_counts.append(X[col].value_counts())
val_counts

- no clinically abnormal values of features

---
Checking for outliers in numerical features

In [ ]:
num_cols = ['resting_blood_pressure', 'serum_cholesterol_mg_per_dl', 'oldpeak_eq_st_depression', 'age',	'max_heart_rate_achieved']
for col in num_cols:
    plt.figure(figsize=(6,4))
    ax = sns.boxplot(data=X, x=col)
    ax.set_title(f"Distribution of {col}")
    plt.tight_layout()
    plt.show()
    plt.close()

- As it is a clinical dataset and the extreme values detected dont seem to be caused by error, these values will not be considered as outliers.
- no extreme outliers, logistic regression is plausible

---
Correlation heatmap to check pairwise collinearities between features and b/w features and target variable

In [ ]:
plt.figure(figsize=(6,5))
sns.heatmap(df[num_cols + [target_col]].corr(), annot=True)
plt.title("Numeric feature correlation")
plt.tight_layout()
plt.show()

- moderate correlations of oldpeak_eq_st_depression and max_heart_rate_achieved to target suggests linear seperability
- no particularly strong inter-feature correlations, but L2-regularization or elastic net reccomended

---
Descriptive statistics of numerical features

In [ ]:
X[num_cols].describe()

---
Prevalence by sex x age_band

In [ ]:
df["age_band"] = pd.cut(df["age"], bins=[0,40,50,60,70,120], labels=["<=40","41-50","51-60","61-70",">70"], include_lowest=True)
pivot_prev = df.groupby(["sex","age_band"])[target_col].mean().unstack()
print(pivot_prev)

plt.figure(figsize=(8,4))
sns.heatmap(pivot_prev, annot=True, fmt=".2f", cmap="Reds")
plt.title("Prevalence heatmap by sex x age_band")
plt.show()

- add sex x age as a new feature as increase in age increases risk more prominently in males

---
Add interaction feature, sex x age as a new feature

In [ ]:
X["sex_age"] = X["sex"].astype(float) * X["age"].astype(float)
df["sex_age"] = df["sex"].astype(float) * df["age"].astype(float)
num_cols = ['resting_blood_pressure', 'serum_cholesterol_mg_per_dl', 'oldpeak_eq_st_depression', 'age',	'max_heart_rate_achieved', 'sex_age']
X

---
Distribution of Numerical Columns

In [ ]:
ncols = 3
nrows = int(np.ceil(len(num_cols)/ncols))
plt.figure(figsize=(ncols*4, nrows*3.2))
for i, c in enumerate(num_cols, 1):
    ax = plt.subplot(nrows, ncols, i)
    sns.histplot(df[c], kde=True, ax=ax, bins=30)
    ax.set_title(c)
plt.tight_layout()
plt.show()

- not heavily skewed, no need of transformation of features

---
Distribution of Categorical Columns

In [ ]:
cat_cols = ['thal', 'chest_pain_type', 'resting_ekg_results']
ordinal_int_cols = ["slope_of_peak_exercise_st_segment", "num_major_vessels"]
binary_cols = ["sex", "exercise_induced_angina", "fasting_blood_sugar_gt_120_mg_per_dl"]

all_cat_like = cat_cols + ordinal_int_cols + binary_cols

# Plot count distributions
ncols = 3
nrows = int(np.ceil(len(all_cat_like)/ncols))
plt.figure(figsize=(ncols*4.5, nrows*3.2))

for i, col in enumerate(all_cat_like, 1):
    ax = plt.subplot(nrows, ncols, i)
    order = df[col].value_counts(dropna=False).index
    sns.countplot(x=col, data=df, order=order, ax=ax)
    ax.set_title(f"{col} distribution")
    ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

- rare categories exist, set handle_unknown to Ignore during one hot encoding, to prevent unseen-level issues

---
KDEs per class

In [ ]:
ncols = 3
nrows = int(np.ceil(len(num_cols)/ncols))
plt.figure(figsize=(ncols*4.5, nrows*3.2))
for i, c in enumerate(num_cols, 1):
    ax = plt.subplot(nrows, ncols, i)
    sns.kdeplot(data=df, x=c, hue=target_col, common_norm=False, fill=True, alpha=0.3, ax=ax)
    ax.set_title(f"{c} by class")
plt.tight_layout()
plt.show()


- overlapping histograms, non linear models to work better.

---
Point-biserial correlations (Pearson between numeric and binary target)

In [ ]:
corrs = {}
for c in num_cols + ordinal_int_cols + binary_cols:
    corrs[c] = np.corrcoef(df[c], df[target_col])[0,1]
corr_s = pd.Series(corrs).sort_values(key=lambda x: np.abs(x), ascending=False)
print("\nPoint-biserial correlations with target:")
print(corr_s.round(3))

plt.figure(figsize=(6, max(3, 0.35*len(corr_s))))
sns.barplot(x=corr_s.values, y=corr_s.index, palette="viridis")
plt.title("Absolute correlation with target")
plt.xlabel("Correlation")
plt.show()

- none are highly correlated to target, adding polynomial/ interaction features for linear models is reccomended

---
Fitting Polynomial features

In [ ]:
poly_interactions = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)

---
Checking prevalance by category

In [ ]:
def risk_table(cat_col):
    tab = (df.groupby(cat_col)[target_col]
             .agg(['count','mean'])
             .rename(columns={'mean':'prevalence'})
             .sort_values('prevalence', ascending=False))
    return tab

for c in ["thal", "chest_pain_type", "slope_of_peak_exercise_st_segment",
          "resting_ekg_results", "num_major_vessels", "exercise_induced_angina"]:
    print(f"\nRisk table by {c}:")
    print(risk_table(c).round(3))

- suggests use of OHE for non-monotone pattern and ordinal encoding for monotone pattern in prevalance

In [ ]:
plt.figure(figsize=(8,4))
thal_prev = df.groupby("thal")[target_col].mean().reset_index()
sns.barplot(x="thal", y=target_col, data=thal_prev)
plt.ylabel("Prevalence")
plt.title("Heart disease prevalence by thal")
plt.show()

- thal set to ordinal encoding

In [ ]:
thal_order = [["normal", "fixed_defect", "reversible_defect"]]
other_cat_cols = [c for c in cat_cols if c != "thal"]

---
Majority baseline accuracy and confusion if predict all 0

In [ ]:
maj_acc = (df[target_col] == 0).mean()
cm_all_zero = np.array([[ (df[target_col]==0).sum(), 0],
                        [ (df[target_col]==1).sum(), 0]])
print(f"\nMajority-class accuracy (all negatives): {maj_acc:.3f}")
print("Confusion matrix if predict all 0 [TN FP; FN TP]:")
print(cm_all_zero)

---
Quick leakage scan: ensure target not trivially derivable from any single column

In [ ]:
max_group_prev = df.drop(columns=[id_col]).groupby(target_col).mean(numeric_only=True)
print("\nGroup means by target (numeric only) to spot suspicious leaks:")
max_group_prev.round(3)

- no leakage found

Task 1: Data Analysis Report
---

---
Data Overview

- The data contained 14 columns; patient_id is a random identifier, and the remaining 13 fields include demographics, vitals, ECG-derived variables, and thallium-test-derived thal categories
- Target variable: heart_disease_present is binary with 0 = no disease and 1 = disease
- Class balance: Distribution is 100 negatives vs 80 positives (≈44% prevalence), indicating moderate imbalance

---
Exploratory Data Analysis

- No duplicates by patient_id; no missing values across features or target; the schema mixes categorical, binary, integer, and float types; categorical levels present as expected.
- Numeric distributions: systolic blood pressure , cholesterol , age , max heart rate , oldpeak ; some extremes exist but were retained given clinical requirement and to avoid biasing model coverage of high-risk profiles.
- Prevalence by sex × age band shows rising risk with age and higher prevalence in males
- Correlation heatmap suggests moderate correlations of oldpeak_eq_st_depression and max_heart_rate_achieved to target suggests linear seperability; no particularly strong inter-feature correlations, but L2-regularization or elastic net reccomended
- Distribution of Numerical Columns suggest that the features are not heavily skewed; no need of any transformation.
- Distribution of Categorical Columns suggest that rare categories exist, prevention of unseen-level issues is necessary
- KDEs per class show overlapping histograms; non linear models tend to work better.
- Pearson correlations between numeric and binary target suggest adding polynomial/ interaction features for linear models.
- Risk tables rank categories (e.g., thal levels and chest pain types) by disease prevalence, revealing high and low risk groups.
- Majority-class “all negative” baseline accuracy is computed to anchor expectations; confusion matrix for all-zero predictions shows the expected behavior and motivates threshold-agnostic metrics for model selection.
- A quick leakage scan via grouping means by target for numeric columns confirms no single trivial proxy of the label

---
Train-test split

In [ ]:
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y , test_size = 0.2, stratify = y, random_state = 22)
X_train, X_thresh, y_train, y_thresh = train_test_split(X_tmp, y_tmp, test_size=0.25, stratify=y_tmp, random_state=22)

- X_train used for training and cross-validation for AUPRC tuning, X_thresh used for threshold tuning

---
Preprocessing pipelines for numerical and categorical features, followed by column transformer to apply the preprocessing pipelines, followed by the model pipelines

In [ ]:
numeric_preprocess = Pipeline(steps=[
    ('scaler', StandardScaler())
])

numeric_preprocess_linear = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('poly', poly_interactions)
])

thal_ordinal = Pipeline(steps=[
    ('ord', OrdinalEncoder(categories=thal_order, handle_unknown='use_encoded_value', unknown_value=-1))
])

other_categorical_preprocess = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_preprocess, num_cols),
        ('thal_ord', thal_ordinal, ['thal']),
        ('cat', other_categorical_preprocess, other_cat_cols),
    ],
    remainder='passthrough'
)

preprocessor_linear = ColumnTransformer(
    transformers=[
        ('num', numeric_preprocess_linear, num_cols),
        ('thal_ord', thal_ordinal, ['thal']),
        ('cat', other_categorical_preprocess, other_cat_cols),
    ],
    remainder='passthrough'
)

log_reg = Pipeline(steps=[
    ('preprocessor', preprocessor_linear),
    ('clf', LogisticRegression(max_iter=500, class_weight='balanced'))
])

rf = Pipeline(steps = [
    ('preprocessor', preprocessor),
    ("clf", RandomForestClassifier(class_weight = None, n_jobs = -1, random_state = 2))
])

#Linear SVM with calibration to get probabilities for AUPRC
# CalibratedClassifierCV wraps LinearSVC; probability-like outputs via sigmoid
base_svc = LinearSVC(class_weight="balanced", max_iter=5000)
cal_svc = CalibratedClassifierCV(base_svc, method="sigmoid", cv=3)
svc = Pipeline([
    ("preprocessor", preprocessor_linear), 
    ("clf", cal_svc)
])

#Define the boosted tree pipeline
hgb_clf = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_depth=None,
    max_bins=255,
    l2_regularization=0.0,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=2
)
# Wrap with calibration to get well-behaved probabilities for AUPRC-driven selection and thresholding
cal_hgb = CalibratedClassifierCV(
    estimator=hgb_clf,
    method="sigmoid",
    cv=3
)
hgb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('clf', cal_hgb)
])

---
Parameter Grids through which hyperparameter tuning is done

In [ ]:
param_grid_logreg = {
    'clf__penalty': ['l2'],
    'clf__C': [0.001, 0.01, 0.1, 1, 10],
    'clf__solver': ['liblinear', 'lbfgs']
}

param_grid_rf = {
    "clf__n_estimators": [200, 400],
    "clf__max_depth": [None, 4, 6, 8],
    "clf__min_samples_split": [2, 5, 10],
    "clf__min_samples_leaf": [1, 2, 4],
}

param_grid_svc = {
    "clf__estimator__C": np.logspace(-3, 3, 7),
}

param_grid_hgb = {
    # Base estimator parameters are addressed under 'clf__base_estimator__<param>'
    'clf__estimator__learning_rate': [0.03, 0.05, 0.1],
    'clf__estimator__max_depth': [None, 3, 5],
    'clf__estimator__min_samples_leaf': [10, 20, 30],
    'clf__method': ['sigmoid'],  # keep fixed unless data is very small; then try 'sigmoid'
}

---
Stratified k-fold Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=2)

---
Function that performs exhaustive hyperparameter tuning with K-fold stratified cross-validation using average_precision as the scorer, refits on the full training split with the best params, and returns the fitted search containing cv_results_ and best_estimator_

In [ ]:
def run_search(pipeline, param_grid, X_tr, y_tr, scoring="average_precision"):
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring=scoring,   # threshold-agnostic metric for selection
        cv=cv,
        n_jobs=-1,
        refit=True,
        verbose=0,
        return_train_score = False
    )
    search.fit(X_tr, y_tr)
    return search

---
Perform hyperparameter tuning on the different models

In [ ]:
search_logreg = run_search(log_reg, param_grid_logreg, X_train, y_train)
search_rf = run_search(rf, param_grid_rf, X_train, y_train)
search_svc = run_search(svc, param_grid_svc, X_train, y_train)
search_hgb = run_search(hgb, param_grid_hgb, X_train, y_train)

---
Compare by best CV score

In [ ]:
candidates = [
    ("LogisticRegression", search_logreg),
    ("RandomForest", search_rf),
    ("SupportVectorClassifier", search_svc),
    ("HistGradientBoosting", search_hgb)
]
best_name, best_search = max(candidates, key=lambda tup: tup[1].best_score_)
print(f"Best model by CV AUPRC: {best_name} (score={best_search.best_score_:.4f})")

best_model = best_search.best_estimator_

In [ ]:
print(best_model)

In [ ]:
fn = best_model.named_steps["preprocessor"].get_feature_names_out()
fn

In [ ]:
rf_clf = best_model.named_steps["clf"]
rf_imp = pd.DataFrame({
    "feature": fn,
    "importance": rf_clf.feature_importances_
}).sort_values("importance", ascending=False).reset_index(drop=True)

print("\nRandomForest impurity-based importances:")
print(rf_imp.to_string(index=False))

In [ ]:
plt.figure(figsize=(7, 6))
sns.barplot(data=rf_imp, x="importance", y="feature", palette="viridis")
plt.title("RandomForest feature importance (impurity-based)")
plt.tight_layout()
plt.show()

---
Function that picks best Threshold value that achieves target recall while maintaining good precision

In [ ]:
def pick_threshold_at_recall_target(y_true, y_prob, recall_target=0.85):
    """Pick the largest threshold that achieves at least the desired recall.
    If none achieves the target, pick the threshold that maximizes F1."""
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_prob)
    # Find thresholds achieving recall >= target
    feasible = [thr for r, thr in zip(recalls[1:], thresholds) if r >= recall_target]
    if len(feasible) > 0:
        return float(max(feasible))  # largest threshold meeting recall target
    # fallback: maximize F1
    f1s = []
    for p, r in zip(precisions[1:], recalls[1:]):
        if p + r > 0:
            f1s.append(2*p*r/(p+r))
        else:
            f1s.append(0)
    best_idx = int(np.argmax(f1s))
    return float(thresholds[best_idx])

---
Choose Threshold

In [ ]:
RECALL_TARGET = 0.8
y_prob_thresh = best_model.predict_proba(X_thresh)[:, 1]
opt_threshold = pick_threshold_at_recall_target(y_thresh, y_prob_thresh, RECALL_TARGET)
print(f"Chosen threshold (target recall {RECALL_TARGET:.2f}): {opt_threshold:.4f}")

---
Training Performance

In [ ]:
#y_prob_train = best_model.predict_proba(X_train)[:, 1]
#y_pred_train = (y_prob_train >= opt_threshold).astype(int)

y_prob_tmp = best_model.predict_proba(X_tmp)[:, 1]
y_pred_tmp = (y_prob_tmp >= opt_threshold).astype(int)

#y_prob_test = best_model.predict_proba(X_test)[:, 1]
#y_pred_test = (y_prob_test >= opt_threshold).astype(int)

auroc = roc_auc_score(y_tmp, y_prob_tmp)
auprc = average_precision_score(y_tmp, y_prob_tmp)
#auroc = roc_auc_score(y_thresh, y_prob_thresh)
#auprc = average_precision_score(y_thresh, y_prob_thresh)
#auroc = roc_auc_score(y_test, y_prob_test)
#auprc = average_precision_score(y_test, y_prob_test)

tn, fp, fn, tp = confusion_matrix(y_tmp, y_pred_tmp).ravel()
#tn, fp, fn, tp = confusion_matrix(y_thresh, y_pred_thresh).ravel()
#tn, fp, fn, tp = confusion_matrix(y_test, y_pred_test).ravel()
eps = 1e-12
accuracy = (tp + tn) / (tp + tn + fp + fn + eps) #Accuracy
recall = tp / (tp + fn + eps)         # Sensitivity
specificity = tn / (tn + fp + eps)    # True negative rate
precision = tp / (tp + fp + eps)      # PPV
npv = tn / (tn + fn + eps)            # NPV
f1 = 2 * precision * recall / (precision + recall + eps)

print("\n=== Metrics (fixed threshold) ===")
print(f"Accuracy: {accuracy:.3f}")
print(f"AUROC: {auroc:.3f}")
print(f"AUPRC (Average Precision): {auprc:.3f}")
print(f"Recall (Sensitivity): {recall:.3f}")
print(f"Specificity: {specificity:.3f}")
print(f"Precision (PPV): {precision:.3f}")
print(f"NPV: {npv:.3f}")
print(f"F1: {f1:.3f}")
print("\nConfusion Matrix [tn, fp; fn, tp]:")
print(np.array([[tn, fp], [fn, tp]]))

---
Threshold Data Performance

In [ ]:
y_pred_thresh = (y_prob_thresh >= opt_threshold).astype(int)
auroc = roc_auc_score(y_thresh, y_prob_thresh)
auprc = average_precision_score(y_thresh, y_prob_thresh)

tn, fp, fn, tp = confusion_matrix(y_thresh, y_pred_thresh).ravel()
eps = 1e-12
accuracy = (tp + tn) / (tp + tn + fp + fn + eps) #Accuracy
recall = tp / (tp + fn + eps)         # Sensitivity
specificity = tn / (tn + fp + eps)    # True negative rate
precision = tp / (tp + fp + eps)      # PPV
npv = tn / (tn + fn + eps)            # NPV
f1 = 2 * precision * recall / (precision + recall + eps)

print("\n=== Threshold Metrics (fixed threshold) ===")
print(f"Accuracy: {accuracy:.3f}")
print(f"AUROC: {auroc:.3f}")
print(f"AUPRC (Average Precision): {auprc:.3f}")
print(f"Recall (Sensitivity): {recall:.3f}")
print(f"Specificity: {specificity:.3f}")
print(f"Precision (PPV): {precision:.3f}")
print(f"NPV: {npv:.3f}")
print(f"F1: {f1:.3f}")
print("\nConfusion Matrix [tn, fp; fn, tp]:")
print(np.array([[tn, fp], [fn, tp]]))

---
Unbiased evaluation on the 20% test split

In [ ]:
y_prob_test = best_model.predict_proba(X_test)[:, 1]
y_pred_test = (y_prob_test >= opt_threshold).astype(int)

In [ ]:
np.size(y_pred_test)

In [ ]:
auroc = roc_auc_score(y_test, y_prob_test)
auprc = average_precision_score(y_test, y_prob_test)

In [ ]:
# Confusion matrix derived metrics
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_test).ravel()
eps = 1e-12
accuracy = (tp + tn) / (tp + tn + fp + fn + eps) #Accuracy
recall = tp / (tp + fn + eps)         # Sensitivity
specificity = tn / (tn + fp + eps)    # True negative rate
precision = tp / (tp + fp + eps)      # PPV
npv = tn / (tn + fn + eps)            # NPV
f1 = 2 * precision * recall / (precision + recall + eps)

In [ ]:
print("\n=== Unbiased Test Metrics (fixed threshold) ===")
print(f"Accuracy: {accuracy:.3f}")
print(f"AUROC: {auroc:.3f}")
print(f"AUPRC (Average Precision): {auprc:.3f}")
print(f"Recall (Sensitivity): {recall:.3f}")
print(f"Specificity: {specificity:.3f}")
print(f"Precision (PPV): {precision:.3f}")
print(f"NPV: {npv:.3f}")
print(f"F1: {f1:.3f}")
print("\nConfusion Matrix [tn, fp; fn, tp]:")
print(np.array([[tn, fp], [fn, tp]]))

---
Baseline

In [ ]:
y_prob_test_baseline = np.array(36*[0])
y_pred_test_baseline = np.array(36*[0])

In [ ]:
auroc = roc_auc_score(y_test, y_prob_test_baseline)
auprc = average_precision_score(y_test, y_prob_test_baseline)

In [ ]:
# Confusion matrix derived metrics
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_test_baseline).ravel()
eps = 1e-12
accuracy = (tp + tn) / (tp + tn + fp + fn + eps) #Accuracy
recall = tp / (tp + fn + eps)         # Sensitivity
specificity = tn / (tn + fp + eps)    # True negative rate
precision = tp / (tp + fp + eps)      # PPV
npv = tn / (tn + fn + eps)            # NPV
f1 = 2 * precision * recall / (precision + recall + eps)

In [ ]:
print("\n=== Unbiased Test Metrics Baseline (fixed threshold) ===")
print(f"Accuracy: {accuracy:.3f}")
print(f"AUROC: {auroc:.3f}")
print(f"AUPRC (Average Precision): {auprc:.3f}")
print(f"Recall (Sensitivity): {recall:.3f}")
print(f"Specificity: {specificity:.3f}")
print(f"Precision (PPV): {precision:.3f}")
print(f"NPV: {npv:.3f}")
print(f"F1: {f1:.3f}")
print("\nConfusion Matrix [tn, fp; fn, tp]:")
print(np.array([[tn, fp], [fn, tp]]))

Executive Summary
---

- The workflow on 180 patients (100 negative, 80 positive) produced a calibrated, threshold-tuned Random Forest pipeline as the best model by cross-validated AUPRC and test evaluation, with optimization for recall due to medical relevance.

- On a held-out test split with threshold set to achieve at least 0.80 recall (due to small dataset) on a validation subset, the model achieved AUROC 0.953, AUPRC 0.943, recall 0.937, precision 0.750, specificity 0.750, and F1 0.833, prioritizing sensitivity with acceptable false positives.

Task 2: Predictive Modelling
---

- Data is split into train, threshold-tuning, and test sets with stratification; numeric features are standardized while categorical features are one-hot encoded and 'thal' column ordinally encoded with handle_unknown='ignore' to ensure robust inference on unseen levels.
- Preprocessing also includes Numerical features to polynomial interaction expansion only for the linear pipeline, keeping trees free to learn interactions intrinsically
- Pipelines encapsulate preprocessing and estimator for clean cross-validation and to prevent data leakage across folds.

---
Candidate Models

- Logistic Regression: balanced class weights, L2 penalty, and solvers liblinear/lbfgs tuned via grid on C; it is interpretable and efficient.
- Random Forest: tuned over n_estimators, max_depth, min_samples_split, and min_samples_leaf; offers non-linear interactions and feature importance.
- Linear SVM with calibration: LinearSVC is wrapped in CalibratedClassifierCV (sigmoid), enabling probability outputs suitable for AUPRC optimization; C is tuned on a log grid.
- HistGradientBoosting also calibrated, with calibration set to sigmoid; this ensures well-behaved probabilities for AUPRC selection and clinically interpretable risk scores.

---
Model selection and thresholding

- Hyperparameters are tuned via StratifiedKFold CV with scoring set to average_precision, which is more clinically relevant than accuracy; the best model by mean CV-AUPRC is then refit on training data for threshold tuning.
- A recall-targeted threshold is chosen on a threshold split: pick the largest threshold that achieves recall ≥ target (e.g., 0.80), falling back to F1-maximizing threshold if the target is unreachable; this seperates model choice from operating point and yields a transparent sensitivity/recall commitment

---
Final evaluation

- The final fixed threshold is applied to the untouched test set to report AUROC, AUPRC, sensitivity (recall), specificity, PPV (precision), NPV, F1, and the confusion matrix
- Key drivers of risk: The strongest contributors were oldpeak (ST depression), the sex×age interaction, thallium test result, and chest pain type 4
- Actionable clinical signals: Exercise-induced angina, maximum heart rate achieved, cholesterol, ST-segment slope, and number of major vessels had substantial importance
- Lower-influence features: Resting EKG categories and fasting blood sugar contributed relatively little in this model configuration

Task 3: Hospital implementation
---

- Operationalizing the Random Forest model requires sensitivity priorities, continuous monitoring, and clear SOPs for high-risk flags; a recall ≥ 0.80 target is appropriate for screening, maintain a setup with Logistic Regression for transparency.
- Governance should include monthly calibration and sensitivity monitoring, a false-negative audit workflow.
- Clinical workflow: For patients flagged at or above the operating threshold, route to targeted follow-ups such as stress testing or imaging.
- Data drift and maintenance: Track prevalence, input distributions, and model score drift; re-estimate thresholds if prevalence changes materially

Task 4: Model comparison report
---

- Candidate models were tuned with 5‑fold stratified CV on Average Precision to emphasize positive‑class ranking quality under potential imbalance; this is preferable to accuracy for screening problems where false negatives are costlier.
- After selection by CV‑AUPRC, a threshold was chosen to meet a recall target on a separate threshold split and locked for final test evaluation; this yields clinically meaningful sensitivity with reportable specificity and PPV/NPV.
- A calibrated, threshold-tuned Random Forest pipeline offers robust performance with minimal feature engineering and is the reccomended model.
- Retain Logistic Regression as a transparent baseline; its coefficients and interactions can assist in clinical validation and model governance.

Task 5: Challenges faced and resolutions
---

- Mixed data types: Numeric, binary, ordinal, and nominal variables required tailored preprocessing; resolved via ColumnTransformer with StandardScaler for numeric, OrdinalEncoder for thal, and OneHotEncoder for nominal, avoiding inappropriate transformations and improving model fit.
- Probability needs for SVM and boosted trees: Certain estimators do not provide calibrated probabilities natively; addressed with CalibratedClassifierCV (sigmoid) to stabilize probabilities for AUPRC‑based selection and thresholding.
- Threshold selection bias: Choosing thresholds on the test set inflates performance; split off a threshold‑tuning set to select the operating threshold, then held out test for unbiased evaluation.
- Class imbalance considerations: Accuracy is misleading; used AUPRC and recall targeting to prioritize detection of true positives, reflecting clinical risk tolerance for missed cases
- Linear models underperform when important risk depends on interactions (e.g., sex x age), Engineered domain-plausible interactions like sex×age and added PolynomialFeatures (interaction_only) only in the linear pipeline